UK Companies House Data Extraction

**Objective:** The goal of this project is to build an automated data pipeline that extracts, parses, and cleans business information from the UK Government's Companies House public register. 

**Tools Used:**
1. **Scrapy:** Navigating the search index to harvest initial profile URLs.
2. **Requests & BeautifulSoup (BS4):** Fetching static HTML and parsing core text data (rapid extraction).
3. **Selenium:** Simulating a real browser to perform dynamic page interactions (clicking tabs).
4. **Python Regular Expressions (re) & Pandas:** Cleaning unstructured text and validating formats.

### Step 0: Environment Setup
Importing all necessary libraries for scraping, parsing, and data manipulation.

In [9]:
import os
import sys
import subprocess
import json
import time
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

### Step 1: Initial Link Harvesting

In this stage, a scrapy spider is deployed to gather individual company profile URLs from the search results. 

**Technical Challenge & Anti-Bot Measures:**
The companies house server employs advanced anti-scraping mechanisms. While the crawler is correctly configured to locate the pagination button, **id="next-page"**, the server actively hides this DOM element from static non-browser requests to prevent bulk downloading. Furthermore, deep navigation is restricted by **robots.txt** policies. 

**Resolution:** To maintain the integrity of the project and adhere to ethical scraping limits, the scrapy spider is utilized to harvest a representative sample of 20 active companies from the initial static response. This baseline sample is perfectly sufficient to demonstrate the full technical pipeline (extraction, interaction, and regex cleaning) in the subsequent steps without risking IP blacklisting.

In [10]:
scraper_code = '''
import scrapy
from scrapy.crawler import CrawlerProcess

url = 'https://find-and-update.company-information.service.gov.uk/advanced-search/get-results?registeredOfficeAddress=London&status=active'

process = CrawlerProcess(settings={
    'LOG_LEVEL': 'ERROR',
    'FEEDS': {
        'outputs/ch_links_output.json': {
            'format': 'json',
            'overwrite': True,
        },
    },
    
    'DOWNLOAD_DELAY': 2,
    'USER_AGENT': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
})

class CompaniesHouseSpider(scrapy.Spider):
    name = 'ch_links'
    start_urls = [url]

    def parse(self, response):
        links = response.css('a.govuk-link::attr(href)').getall()
        
        for link in links:
            if '/company/' in link:
                yield {
                    'company_url': response.urljoin(link)
                }

process.crawl(CompaniesHouseSpider)
process.start()
'''

os.makedirs("scripts", exist_ok=True)
os.makedirs("outputs", exist_ok=True) 

with open("scripts/ch_scraper.py", "w", encoding="utf-8") as file:
    file.write(scraper_code)

script_path = r'scripts/ch_scraper.py'

result = subprocess.run([sys.executable, script_path], capture_output=True, text=True)

print("Scraping is done")
if result.stderr:
    print(result.stderr)

Scraping is done


### Step 2: Static Parsing and Dynamic Interaction

To optimize extraction speed while handling dynamic web elements, this stage splits the workload between static parsing and dynamic interaction:

1. **Static HTML Parsing (Requests + BeautifulSoup):**
   Standard HTTP requests are sent to each extracted URL, utilizing custom user-agent headers to bypass basic WAF (Web Application Firewall) blocks. The raw HTML is immediately parsed using BeautifulSoup. This is highly efficient for extracting core structural data: Company Name, Number, Status, Incorporation Date, and Raw Address. String manipulation is applied to clean specific tags.

2. **Dynamic Interaction (Selenium WebDriver):**
   Selenium initializes a headless-capable browser session for the same URL. The webdriver targets and clicks the "People" tab, **id="people-tab"**, simulating legitimate user behavior to handle dynamic content rendering that static requests cannot execute.

In [11]:
with open('outputs/ch_links_output.json', 'r', encoding='utf-8') as f:
    links_data = json.load(f)

scraped_data = []

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9',
}

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

for item in links_data:
    url = item['company_url']
    time.sleep(1)
    
    try:
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, 'html.parser')
        
        page_title = soup.title.get_text(strip=True) if soup.title else ""
        raw_name = page_title.split(' - ')[0] if ' - ' in page_title else "NO TITLE"
        company_name = raw_name.replace(' overview', '').strip()

        number_elem = soup.find(id='company-number')
        company_number = number_elem.find('strong').get_text(strip=True) if number_elem and number_elem.find('strong') else None

        status_elem = soup.find(id='company-status')
        company_status = status_elem.get_text(strip=True) if status_elem else None

        date_elem = soup.find(id='company-creation-date')
        incorp_date = date_elem.get_text(strip=True) if date_elem else None

        address_elem = soup.find(string=lambda s: s and "Registered office address" in s)
        if address_elem:
            address_node = address_elem.find_next('dd') or address_elem.find_next('div')
            address = address_node.get_text(separator=", ", strip=True) if address_node else None
        else:
            address = None
            
        driver.get(url)
        time.sleep(1)
        
        try:
            people_tab = driver.find_element(By.ID, "people-tab")
            people_tab.click()
            sel_status = "Click recorded"
        except:
            sel_status = "No tab"

        scraped_data.append({
            'Company Name': company_name,
            'Company Number': company_number,
            'Incorporation Date': incorp_date,
            'Status': company_status,
            'Raw Address': address,
            'Selenium Check': sel_status,
            'Profile URL': url
        })
        
    except Exception as e:
        print(f"Error processing {url}: {e}")

driver.quit()

df_final = pd.DataFrame(scraped_data)
df_final.to_csv('outputs/ch_final_data.csv', index=False)

display(df_final.head())

,Company Name,Company Number,Incorporation Date,Status,Raw Address,Selenium Check,Profile URL
0,CAVELL SOLICITORS LLP,OC305893,28 October 2003,Active,"10-12 Whitechapel Road, London, E1 1EW",Click recorded,https://find-and-update.company-information.se...
1,ERG WIND ITALY 8 LIMITED LIABILITY PARTNERSHIP,OC311128,24 January 2005,Active,"6 St. Andrew Street, London, England, EC4A 3AE",Click recorded,https://find-and-update.company-information.se...
2,LINDMAX LLP,OC311272,29 January 2005,Active,"Suite 1 Level 14 The Broadgate Tower, 20 Primr...",Click recorded,https://find-and-update.company-information.se...
3,TAYLOR HAMPTON LAW LLP,OC332458,29 October 2007,Active,"3rd Floor, 218 Strand, London, WC2R 1AT",Click recorded,https://find-and-update.company-information.se...
4,OC353080 LLP,OC353080,10 March 2010,Active,"87 Devonshire Road, Palmers Green, London, N13...",Click recorded,https://find-and-update.company-information.se...


### Step 3: Data Cleaning and Validation

The raw data extracted from the HTML requires standardization. The python re module was utilized alongside pandas to clean the dataset:

* **UK Postcode Extraction:** The raw address string varies greatly in format. I used **re.findall()** with the pattern **r'[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}** to precisely target and extract the alphanumeric UK postcode.
* **Company Number Validation:** Official UK company numbers follow a specific syntax. I used **re.match()** with the pattern **r'^([A-Z]{2})?\d{6,8}$'** to validate if the extracted strings conform to the official registry format, returning a boolean value.

The final, cleaned dataset is exported to **ch_cleaned_data.csv**.

In [12]:
df_final = pd.read_csv('outputs/ch_final_data.csv')
postcode_pattern = r'[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}'

def extract_postcode(address):
    if not address or pd.isna(address):
        return None
    matches = re.findall(postcode_pattern, str(address).upper())
    return matches[0] if matches else None

df_final['Postcode'] = df_final['Raw Address'].apply(extract_postcode)

number_pattern = r'^([A-Z]{2})?\d{6,8}$'

def validate_number(num):
    if not num or pd.isna(num):
        return False
    return bool(re.match(number_pattern, str(num).strip()))

df_final['Is Number Valid'] = df_final['Company Number'].apply(validate_number)

df_final.to_csv('outputs/ch_cleaned_data.csv', index=False)

display(df_final[['Company Name', 'Raw Address', 'Postcode', 'Company Number', 'Is Number Valid']].head())

,Company Name,Raw Address,Postcode,Company Number,Is Number Valid
0,CAVELL SOLICITORS LLP,"10-12 Whitechapel Road, London, E1 1EW",E1 1EW,OC305893,True
1,ERG WIND ITALY 8 LIMITED LIABILITY PARTNERSHIP,"6 St. Andrew Street, London, England, EC4A 3AE",EC4A 3AE,OC311128,True
2,LINDMAX LLP,"Suite 1 Level 14 The Broadgate Tower, 20 Primr...",EC2A 2EW,OC311272,True
3,TAYLOR HAMPTON LAW LLP,"3rd Floor, 218 Strand, London, WC2R 1AT",WC2R 1AT,OC332458,True
4,OC353080 LLP,"87 Devonshire Road, Palmers Green, London, N13...",N13 4QU,OC353080,True


### Conclusion
The project demonstrates a highly resilient data extraction architecture. By dividing tasks using scrapy for indexing, rRequests/BS4 for rapid static parsing, selenium for dynamic interactions, and regex for precise data validation the pipeline successfully transforms unstructured web data into a clean, analytical-ready dataset while navigating real-world anti-bot protections.